# Part I: Initial Benchmark
We are going to benchmark our current implementation, first we want to set $n$ and $m$ so that they can be on a reasonably sized instance. 

We use the ks_func function we implemented in HW2, and we set $n$  and $m$  to be 1,000,000 instead of 1000 to make the memory bottleneck and performance issues more observable.  And we use the benchmark tool to see the execute time and see what'll happen.



In [ ]:
using Plots
using Revise
includet("C:/Users/Odysseus/GR6104_Homework/HW3/src/ks-stat.jl")

In [ ]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000000)
m = randn(1000000)
@btime ks_func(n,m)
@benchmark ks_func(n,m)


The current implementation takes approximately 170 ms to run. More importantly, it allocates 91.55 MiB of memory. This indicates a significant inefficiency, likely due to the creation of intermediate data structures during the labeling and sorting process.

# Part II: Iterative Code Optimization
## 1. Identify Bottlenecks

In [ ]:
ks_func(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview ks_func(n, m)

## Function and Line Number:
Based on the result from the Flame Graph, the primary bottleneck is located in the ks_func function, specifically at line 10 in ks-stat.jl. This line corresponds to the sort! operation on the combined array.

## Nature of the Bottleneck:
The bottleneck is primarily caused by unnecessary memory allocation, which leads to excessive computation:
    1. Memory Allocation caused by string, since the code creates a new tuple containing string (value,"label") for every single data point for a single run, it causes a huge volume of pressure on system's memory and the Collector.
    2. The Flame Graph shows that 74% of the execution time is spent in sort!. We might need to change the sort! a bit.

# 2. Alternative Implementations:

### Rationale for Alternatives
The bottleneck in the original code was the creation of intermediate `(value, label)` tuples and sorting a mixed array.
* **Alternative 1 (Separate Sorting + Two-Pointer):** Instead of mixing the arrays, we sort `X` and `Y` separately. This allows us to use Julia's highly optimized native float sorting (SIMD-accelerated). We then use a "Two-Pointer" algorithm to calculate the KS statistic in a single linear pass ($O(N)$), avoiding all tuple allocations.

In [7]:



function ks_func_2pt(X::AbstractVector,Y::AbstractVector)
    


    n = randn(X)
    m = randn(Y)
    cdf_x = 0
    cdf_y = 0
    sort_x = sort!(X)
    sort_y =sort!(Y)

    while i <=n & sort![i]
        current_val_x +=1/n
        i+=1
    end

    while j <=m & sort![j]
        current_val_y +=1/m
        j+=1
    end


end        

ks_func_2pt (generic function with 1 method)